In [1]:
import math
import cmath
from functools import lru_cache

# Optional: for classical 6j reference
try:
    import sympy as sp
    from sympy.physics.wigner import wigner_6j
    HAVE_SYMPY = True
except Exception:
    HAVE_SYMPY = False

EPS_THETA = 1e-10


# ==========================
# q-number and q-factorial
# ==========================

def q_number(n: int, theta: float) -> complex:
    """
    [n]_q = sin(n*theta) / sin(theta), q = e^{i theta}
    For small theta, returns n.
    """
    if n < 0:
        raise ValueError("q_number: n must be >= 0")

    # theta ~ 0 or 2π -> classical limit
    if abs(theta) < EPS_THETA or abs(abs(theta) - 2 * math.pi) < EPS_THETA:
        return complex(n)

    # theta ~ pi (q ≈ -1), integer n has removable singularity
    if abs(abs(theta) - math.pi) < EPS_THETA:
        # limit of sin(nθ)/sin θ as θ→π is n*(-1)^(n+1)
        return complex(n * ((-1) ** (n + 1)))

    den = math.sin(theta)
    if abs(den) < 1e-16:
        return complex(n)

    num = math.sin(n * theta)
    return complex(num / den)


@lru_cache(maxsize=None)
def log_q_factorial(n: int, theta: float) -> complex:
    """
    log([n]_q!) as complex a+ib, where [n]_q! = ∏_{k=1}^n [k]_q.
    For theta ≈ 0, uses real lgamma for speed.
    Returns -inf+0j if the product effectively vanishes.
    """
    if n < 0:
        return complex(float("-inf"), 0.0)
    if n == 0:
        return complex(0.0, 0.0)

    # classical limit
    if abs(theta) < EPS_THETA or abs(abs(theta) - 2 * math.pi) < EPS_THETA:
        return complex(math.lgamma(n + 1.0), 0.0)

    log_mag = 0.0
    phase = 0.0
    for k in range(1, n + 1):
        z = q_number(k, theta)
        r = abs(z)
        if r < 1e-30:
            return complex(float("-inf"), 0.0)
        log_mag += math.log(r)
        phase += cmath.phase(z)
    return complex(log_mag, phase)


def safe_exp_log(z: complex) -> complex:
    """
    For z = a+ib, return exp(z).
    If a=-inf, return 0.
    """
    if math.isinf(z.real) and z.real < 0:
        return 0j
    return cmath.rect(math.exp(z.real), z.imag)


# ==========================
# q-triangle coefficient
# ==========================

@lru_cache(maxsize=None)
def log_q_delta(a: float, b: float, c: float, theta: float) -> complex:
    """
    log(Δ_q(a,b,c)), where
    Δ_q(a,b,c) = sqrt( [a+b-c]! [a-b+c]! [-a+b+c]! / [a+b+c+1]! ).
    Enforces SU(2) triangle + parity rules.
    """
    # non-negativity
    if a < 0 or b < 0 or c < 0:
        return complex(float("-inf"), 0.0)

    # triangle inequalities
    if (a + b < c) or (a + c < b) or (b + c < a):
        return complex(float("-inf"), 0.0)

    # parity: a+b+c integer
    if abs((a + b + c) - round(a + b + c)) > 1e-8:
        return complex(float("-inf"), 0.0)

    # factorial arguments (integers)
    n1 = int(round(a + b - c))
    n2 = int(round(a - b + c))
    n3 = int(round(-a + b + c))
    n4 = int(round(a + b + c + 1))

    if min(n1, n2, n3) < 0:
        return complex(float("-inf"), 0.0)

    lm1 = log_q_factorial(n1, theta)
    lm2 = log_q_factorial(n2, theta)
    lm3 = log_q_factorial(n3, theta)
    lm4 = log_q_factorial(n4, theta)

    if any(math.isinf(z.real) and z.real < 0 for z in (lm1, lm2, lm3, lm4)):
        return complex(float("-inf"), 0.0)

    return 0.5 * (lm1 + lm2 + lm3 - lm4)


# ==========================
# q-deformed 6j via Racah
# ==========================

def sixj_q(j1, j2, j3, j4, j5, j6, theta: float) -> complex:
    """
    q-deformed SU(2) 6j symbol with q = e^{i theta}.
    j's are half-integers (floats or ints).
    """
    # 1. prefactor from four triangles
    deltas = [
        log_q_delta(j1, j2, j3, theta),
        log_q_delta(j1, j5, j6, theta),
        log_q_delta(j4, j2, j6, theta),
        log_q_delta(j4, j5, j3, theta),
    ]
    if any(math.isinf(d.real) and d.real < 0 for d in deltas):
        return 0j
    log_pref = sum(deltas)

    # 2. Racah t-range
    t_min = max(
        j1 + j2 + j3,
        j1 + j5 + j6,
        j4 + j2 + j6,
        j4 + j5 + j3,
    )
    t_max = min(
        j1 + j2 + j4 + j5,
        j1 + j3 + j4 + j6,
        j2 + j3 + j5 + j6,
    )
    t_start = int(math.ceil(t_min - 1e-8))
    t_end = int(math.floor(t_max + 1e-8))
    if t_start > t_end:
        return 0j

    sum_val = 0j

    for t in range(t_start, t_end + 1):
        # numerator [t+1]_q!
        ln_num = log_q_factorial(t + 1, theta)
        if math.isinf(ln_num.real) and ln_num.real < 0:
            continue

        # denominator: 7 q-factorials
        args = [
            t - (j1 + j2 + j3),
            t - (j1 + j5 + j6),
            t - (j4 + j2 + j6),
            t - (j4 + j5 + j3),
            (j1 + j2 + j4 + j5) - t,
            (j1 + j3 + j4 + j6) - t,
            (j2 + j3 + j5 + j6) - t,
        ]
        ln_den = complex(0.0, 0.0)
        valid = True
        for a in args:
            n = int(round(a))
            if n < 0:
                valid = False
                break
            ln_f = log_q_factorial(n, theta)
            if math.isinf(ln_f.real) and ln_f.real < 0:
                valid = False
                break
            ln_den += ln_f
        if not valid:
            continue

        # (-1)^t factor: add i*pi*t to phase
        log_term = ln_num - ln_den + complex(0.0, math.pi * t)
        term = safe_exp_log(log_term)
        sum_val += term

    pref = safe_exp_log(log_pref)
    return pref * sum_val


# ==========================
# Classical 6j via sympy (for tests)
# ==========================

def sixj_classical(j1, j2, j3, j4, j5, j6) -> float:
    """
    Classical Wigner 6j using sympy if available.
    j's are half-integers.
    """
    if not HAVE_SYMPY:
        raise RuntimeError("SymPy not available for classical 6j.")
    return float(
        wigner_6j(
            int(round(2 * j1)),
            int(round(2 * j2)),
            int(round(2 * j3)),
            int(round(2 * j4)),
            int(round(2 * j5)),
            int(round(2 * j6)),
        ).evalf()
    )


# ==========================
# Basic tests and scaling
# ==========================

def test_classical_limit():
    if not HAVE_SYMPY:
        print("SymPy not available, skipping classical limit tests.")
        return

    print("== Classical limit tests ==")
    test_cases = [
        (0.5, 0.5, 1.0, 0.5, 0.5, 1.0),
        (0.5, 0.5, 0.0, 0.5, 0.5, 0.0),
        (1.0, 1.0, 1.0, 1.0, 1.0, 1.0),
    ]
    thetas = [1e-3, 5e-3, 1e-2]

    for jtuple in test_cases:
        j1, j2, j3, j4, j5, j6 = jtuple
        exact = sixj_classical(*jtuple)
        print(f"j's = {jtuple}, classical = {exact:.12f}")
        for th in thetas:
            val_q = sixj_q(j1, j2, j3, j4, j5, j6, th)
            err = abs(val_q - exact)
            print(f"  theta={th:.1e}: sixj_q={val_q:.12f}, |Δ|={err:.3e}")
        print()


def test_symmetry():
    if not HAVE_SYMPY:
        print("SymPy not available, skipping symmetry test (still tests q-symmetry numerically).")

    print("== Symmetry test (swap first two columns) ==")
    j = (0.5, 1.0, 1.5, 0.5, 1.0, 1.0)
    j1, j2, j3, j4, j5, j6 = j
    theta = 0.3
    val1 = sixj_q(j1, j2, j3, j4, j5, j6, theta)
    val2 = sixj_q(j2, j1, j3, j5, j4, j6, theta)  # swap cols (1,2) and (4,5)
    print(f"sixj_q{j} = {val1}")
    print(f"sixj_q swapped = {val2}")
    print(f"|difference| = {abs(val1 - val2):.3e}\n")


def scaling_experiment():
    """
    Crude check of |6j_q - 6j| ~ theta^2 * J_max^alpha for one shape family.
    """
    if not HAVE_SYMPY:
        print("SymPy not available, skipping scaling experiment.")
        return

    print("== Scaling experiment ==")
    theta = 0.01
    J_vals = [1, 2, 3, 4]

    diffs = []
    for J in J_vals:
        # generic-ish shape: all j_i = J or J-1, but admissible
        j1 = J
        j2 = J - 1
        j3 = J
        j4 = J - 1
        j5 = J
        j6 = J - 1
        # ensure non-negative
        if j2 < 0 or j4 < 0 or j6 < 0:
            continue
        jtuple = (float(j1), float(j2), float(j3),
                  float(j4), float(j5), float(j6))

        exact = sixj_classical(*jtuple)
        val_q = sixj_q(*jtuple, theta)
        diff = abs(val_q - exact)
        diffs.append((J, diff))
        print(f"J_max={J}, sixj={exact:.3e}, sixj_q={val_q:.3e}, |Δ|={diff:.3e}")

    if len(diffs) >= 2:
        # crude log-log fit: diff ~ C * J^alpha
        import numpy as np
        J_arr = np.array([d[0] for d in diffs], dtype=float)
        D_arr = np.array([d[1] for d in diffs], dtype=float)
        # avoid zeros
        mask = D_arr > 0
        J_arr = J_arr[mask]
        D_arr = D_arr[mask]
        if len(J_arr) >= 2:
            logJ = np.log(J_arr)
            logD = np.log(D_arr)
            # Simple linear regression (logD = alpha * logJ + logC)
            A = np.vstack([logJ, np.ones(len(logJ))]).T
            alpha, logC = np.linalg.lstsq(A, logD, rcond=None)[0]
            print(f"\nEstimated scaling exponent (alpha): {alpha:.2f}")
            print(f"Expected: |6j_q - 6j| ~ theta^2 * J_max^alpha, so alpha is around 4-6.")



SyntaxError: incomplete input (ipython-input-2792875221.py, line 314)